

##  Optimizing Solar Energy Use

This Project is done as a part of Opti 201 course given By Gurobi. This example draws inspiration from the [IEEE Predict+Optimize Technical Challenge (2021)](https://ieee-dataport.org/competitions/ieee-cis-technical-challenge-predictoptimize-renewable-energy-scheduling),


### Objective

The goal is to minimize electricity costs over a 6-day period by reducing reliance on grid power. We aim to meet a building's electricity demand using a combination of:

* Forecasted solar energy production,
* Two on-site battery storage systems, and
* Grid electricity as a backup.

### Decision Variables

Every 30-minute time step involves three key decisions:

1. How much electricity to purchase from the grid,
2. How much to charge or discharge Battery 1,
3. How much to charge or discharge Battery 2.

### Model Output

The optimization model produces:

* A charging/discharging schedule for both batteries,
* A plan indicating when and how much electricity to purchase from the grid.

Forecasted electricity demand is generated using [Prophet](https://facebook.github.io/prophet/), an open-source forecasting tool developed by Meta.



### Formulation

#### 📘 Sets and Input Parameters

We define the following sets and parameters for the optimization model:

---

**Sets**

* **Batteries**:
  $b \in B = \{\texttt{Battery0}, \texttt{Battery1}\}$
  *Set of available battery storage units.*

* **Time Periods**:
  $t \in T = \{0, 1, \dots, 179\}$
  *Each time step represents a 30-minute interval over six days (Monday to Saturday, October 2020).*

---

**Input Parameters**

* **Battery Capacity**:
  $c_b$: Maximum storage capacity of battery $b \in B$
  → `capacity[b]`

* **Charging Efficiency Loss**:
  $p_b$: Fractional loss during energy transfer into battery $b \in B$
  → `p_loss[b]`

* **Initial Battery Charge**:
  $q_b$: Initial energy level stored in battery $b \in B$
  → `initial[b]`

* **Solar Generation**:
  $\text{solar}_t$: Forecasted solar power generation at time $t \in T$
  → `solar_values[t]`

* **Total Energy Demand**:
  $d_t$: Combined building and classroom electricity demand at time $t \in T$
  → `total_demand[t]`



#### Decision Variables
Let $f^{in}_{b,t}$ be the amount each battery, $b$, `charges` at time period,  $t$, $\forall b\in B, t\in T$. $\quad\quad \texttt{flow}\_\texttt{in[b,t]}$

Let $f^{out}_{b,t}$ be the amount each battery `discharges`, similarly. $\quad\quad \texttt{flow}\_\texttt{out[b,t]}$

The max amount that each battery can charge or discharge in a single period to be 20 kW.

$s_{b,t}$ is the current amount of energy in battery, $b$, at the end of time period, $t, \forall b\in B, t\in T$. $\quad\texttt{state[b,t]}$

$gen_{t}$ is the amount of available solar energy that is used in period $t$, $\forall t \in T$. $\quad\texttt{gen[t]}$

$grid_{t}$ : This variable indicates the amount of energy purchased from the grid at time period, $t$, $\forall t \in T$.  $\quad\texttt{grid[t]}$

We are not considering the ability to "sell back" electricity, so $grid_t \ge 0$.


#### ⚙️ Model Constraints

The model is subject to the following constraints to ensure feasibility, realism, and operational safety:

---

**1. Power Balance**
At each time step $t \in T$, the building's total energy demand must be met by a combination of:

* Energy discharged from the batteries,
* Solar generation used directly,
* Electricity purchased from the grid.

$$
\sum_{b \in B} \left( f^{\text{out}}_{b,t} - p_b f^{\text{in}}_{b,t} \right) + \text{gen}_t + \text{grid}_t = d_t \quad \forall t \in T
$$

---

**2. Battery State Dynamics**
Tracks the energy level $s_{b,t}$ in each battery over time:

* **Initial Time Step**:

$$
s_{b,0} = q_b + p_b f^{\text{in}}_{b,0} - f^{\text{out}}_{b,0}
$$

* **Subsequent Time Steps**:

$$
s_{b,t} = s_{b,t-1} + p_b f^{\text{in}}_{b,t} - f^{\text{out}}_{b,t} \quad \forall b \in B,\ t \ge 1
$$

---

**3. Solar Availability Constraint**
At any time $t \in T$, the sum of solar energy used to charge batteries and power the building must not exceed the available solar generation:

$$
f^{\text{in}}_{\texttt{Battery0},t} + f^{\text{in}}_{\texttt{Battery1},t} + \text{gen}_t \leq \text{solar}_t \quad \forall t \in T
$$

---

**4. Mutually Exclusive Charge/Discharge Constraint**
A battery cannot charge and discharge in the same time period. We introduce binary variables $z_{b,t} \in \{0,1\}$ to enforce this:

* If $z_{b,t} = 1$, the battery is allowed to charge.
* If $z_{b,t} = 0$, the battery is allowed to discharge.

$$
\begin{aligned}
f^{\text{in}}_{b,t} &\leq 20 \cdot z_{b,t} \quad &\forall b \in B, t \in T \\
f^{\text{out}}_{b,t} &\leq 20 \cdot (1 - z_{b,t}) \quad &\forall b \in B, t \in T
\end{aligned}
$$

> *Note*: This constraint structure may look familiar — it's a common modeling pattern to enforce binary logic over mutually exclusive activities.

---

**5. Battery Capacity Limits**
The state of charge for each battery must remain within its maximum capacity:

$$
s_{b,t} \leq c_b \quad \forall b \in B,\ t \in T
$$





#### 🎯 Objective Function

The goal of the optimization model is to **minimize the total electricity purchased from the grid** over the entire time horizon:

$$
\min \sum_{t \in T} \text{grid}_t
$$

Where:

* $\text{grid}_t$ is the amount of electricity (in energy units) purchased from the grid at time period $t$.

This objective promotes cost-efficiency and sustainable energy use by leveraging available solar power and battery storage to reduce grid dependency.


### Gurobipy Code
#### Sets and Input parameters

In [1]:
# Install required packages
%pip install gurobipy

# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

# Base path for data files
data_path = 'https://raw.githubusercontent.com/Gurobi/modeling-examples/master/optimization101/Modeling_Session_2/'

# Load solar forecast data and round to 3 decimal places
solar_df = pd.read_csv(f'{data_path}pred_solar_values.csv')
solar_values = solar_df['yhat'].round(3).reset_index(drop=True)

# Load demand components: scheduled demand and average building baseline
schedule_df = pd.read_csv(f'{data_path}schedule_demand.csv')
building_df = pd.read_csv(f'{data_path}building_demand.csv')
total_demand = schedule_df['sched_demand'] + building_df['build_demand']

# Display aggregate stats for verification
print(f"Total Solar Generation: {solar_values.sum():.2f} kW")
print(f"Total Demand: {total_demand.sum():.2f} kW")

# Battery configuration
batteries = ["Battery0", "Battery1"]
capacity = {"Battery0": 60, "Battery1": 80}       # Capacity in kW
p_loss   = {"Battery0": 0.95, "Battery1": 0.90}   # Charging efficiency
initial  = {"Battery0": 0, "Battery1": 0}         # Initial charge in kW

# Time periods (30-minute intervals over 6 days)
time_periods = range(len(solar_values))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 42.3 MB/s eta 0:00:00
Total Solar Generation: 4939.17 kW
Total Demand: 5250.60 kW


#### Decison Variables

In [2]:
# Initialize the optimization model
m = gp.Model("Energy_Storage_Optimization")

# Decision variables

# Energy charged into each battery at each time step
flow_in = m.addVars(batteries, time_periods, name="flow_in")

# Energy discharged from each battery at each time step
flow_out = m.addVars(batteries, time_periods, name="flow_out")

# Energy purchased from the grid at each time step
grid = m.addVars(time_periods, name="grid")

# Battery state of charge at each time step (bounded by capacity)
state = m.addVars(
    batteries, time_periods,
    ub={(b, t): capacity[b] for b in batteries for t in time_periods},
    name="state"
)

# Solar energy used directly by the building at each time step
gen = m.addVars(time_periods, name="gen")

# Binary variables to control charge/discharge mode
# zwitch[b,t] = 1 if battery b is charging at time t; 0 if discharging
zwitch = m.addVars(batteries, time_periods, vtype=GRB.BINARY, name="zwitch")


Restricted license - for non-production use only - expires 2026-11-23


#### Constraints

In [4]:
# -----------------------------
# Add Model Constraints
# -----------------------------

# Power balance constraint:
# Supply (battery discharge + solar + grid) must meet demand at every time step.
m.addConstrs(
    (
        gp.quicksum(flow_out[b, t] - p_loss[b] * flow_in[b, t] for b in batteries)
        + gen[t] + grid[t] == total_demand[t]
        for t in time_periods
    ),
    name="power_balance"
)

# Battery state constraints:
# - Initial battery state at t = 0
m.addConstrs(
    (
        state[b, 0] == initial[b] + p_loss[b] * flow_in[b, 0] - flow_out[b, 0]
        for b in batteries
    ),
    name="initial_state"
)

# - State update for all subsequent time periods
m.addConstrs(
    (
        state[b, t] == state[b, t-1] + p_loss[b] * flow_in[b, t] - flow_out[b, t]
        for b in batteries for t in time_periods if t >= 1
    ),
    name="subsequent_states"
)

# Solar generation constraint:
# The total energy drawn from solar (charging + direct use) can't exceed what's available.
m.addConstrs(
    (
        flow_in['Battery0', t] + flow_in['Battery1', t] + gen[t] <= solar_values[t]
        for t in time_periods
    ),
    name="solar_avail"
)

# Mutually exclusive charge/discharge constraint:
# Use binary variable zwitch to prevent simultaneous charging and discharging.
m.addConstrs(
    (
        flow_in[b, t] <= 20 * zwitch[b, t]
        for b in batteries for t in time_periods
    ),
    name="to_charge"
)

m.addConstrs(
    (
        flow_out[b, t] <= 20 * (1 - zwitch[b, t])
        for b in batteries for t in time_periods
    ),
    name="to_discharge"
)


{('Battery0', 0): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 1): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 2): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 3): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 4): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 5): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 6): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 7): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 8): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 9): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 10): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 11): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 12): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 13): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 14): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 15): <gurobi.Constr *Awaiting Model Update*>,
 ('Battery0', 16): <gurobi.Constr *Awaiting Model 

#### 🎯 Objective Function

Our primary goal is to **minimize the total electricity purchased from the grid**. In an extended version of this model, we also explore an alternative objective: **minimizing total cost**, using time-dependent electricity prices.

To enable comparisons across both formulations—total energy vs. total cost—we'll track both the **quantity of electricity purchased** and the **associated cost** in each scenario. This setup allows for deeper analysis and informed decision-making across varying priorities.

In [5]:
# Load expected electricity prices for each time period
avg_price = pd.read_csv(f'{data_path}expected_price.csv')
price = avg_price['price']

# Define expressions for the objective metrics
total_grid = grid.sum()  # Total electricity purchased from the grid
total_cost = gp.quicksum(price[t] * grid[t] for t in time_periods)  # Total cost of grid electricity

# Set the objective to minimize total electricity purchased from the grid (original objective)
m.setObjective(total_grid, GRB.MINIMIZE)


In [7]:
### Solve the Model
m.optimize()

results = pd.DataFrame([[round(v,2) for v in [total_cost.getValue(),total_grid.getValue()]]],
                       columns = ['Cost','GridAmount']
                       )

results

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1440 rows, 1800 columns and 4498 nonzeros
Model fingerprint: 0xaa31bb6a
Variable types: 1440 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [9e-01, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e-01, 9e+01]
Found heuristic solution: objective 5250.6000000
Presolve removed 84 rows and 265 columns
Presolve time: 0.02s
Presolved: 1356 rows, 1535 columns, 4109 nonzeros
Found heuristic solution: objective 4986.0960000
Variable types: 1199 continuous, 336 integer (336 binary)

Root relaxation: objective 1.281678e+03, 597 iterations, 0.02 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  D

,Cost,GridAmount
0,789.79,1281.68




The original goal of the model is to minimize the total electricity drawn from the grid. Alternatively, we can adjust the objective to minimize the total cost by incorporating expected electricity prices for each time period. Next, we’ll load the price data, update the objective accordingly, and resolve the model to compare results.


In [8]:
# Update the objective to minimize total cost instead of grid energy
m.setObjective(total_cost, GRB.MINIMIZE)

# Solve the optimization model
m.optimize()

# Append the results: total cost and total grid energy used
results = pd.concat(
    [
        results,
        pd.DataFrame(
            [[round(total_cost.getValue(), 2), round(total_grid.getValue(), 2)]],
            columns=['Cost', 'GridAmount']
        )
    ],
    ignore_index=True
)

# Display the updated results
results

# Create a copy of the optimized model for future use
mp = m.copy()


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1440 rows, 1800 columns and 4498 nonzeros
Model fingerprint: 0x66bddde2
Variable types: 1440 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [9e-01, 2e+01]
  Objective range  [6e-02, 2e+00]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e-01, 9e+01]

MIP start from previous solve produced solution with objective 607.227 (0.02s)
Loaded MIP start from previous solve with objective 607.227

Presolve removed 84 rows and 265 columns
Presolve time: 0.02s
Presolved: 1356 rows, 1535 columns, 4109 nonzeros
Variable types: 1199 continuous, 336 integer (336 binary)

Root relaxation: objective 6.019761e+02, 754 iterations, 0.04 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bo

### Handling Multiple Objectives

So far, we’ve explored two separate optimization goals: minimizing total grid electricity purchased, and minimizing total cost based on expected prices. To strike a balance between these priorities, Gurobi offers native support for **multi-objective optimization**.

Next, we’ll demonstrate how to combine these objectives using a **weighted sum approach**, allowing us to control the trade-off between minimizing grid usage and minimizing cost in a single model.


#### Weighted Objectives

When combining multiple objectives using a weighted sum, selecting appropriate **weights** is crucial. These weights determine the relative importance of each objective, but choosing them can be challenging due to differences in units and scale between the objectives.

Careful calibration or normalization is often necessary to ensure the weights reflect true priorities and produce meaningful trade-offs.



In [10]:
m.setObjectiveN(total_cost, index=0, weight = 1, name="cost")
m.setObjectiveN(total_grid, index=1, weight = 10, name= "grid")

In [11]:
m.ModelSense = GRB.MINIMIZE
m.optimize()

results = pd.concat([results,
                     pd.DataFrame([[round(v,2) for v in [total_cost.getValue(),total_grid.getValue()]]], columns = ['Cost','GridAmount'])
                    ],
                    ignore_index=True
                    )

results

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1440 rows, 1800 columns and 4498 nonzeros
Model fingerprint: 0x9cf62986
Variable types: 1440 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [9e-01, 2e+01]
  Objective range  [6e-02, 2e+00]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e-01, 9e+01]

---------------------------------------------------------------------------
Multi-objectives: starting optimization with 2 objectives (1 combined)...
---------------------------------------------------------------------------

Multi-objectives: optimize objective 1 (weighted) ...
---------------------------------------------------------------------------

Optimize a model with 1440 rows, 1800 columns and 4498 nonzeros
Model fingerprint: 0xf11f89cc


,Cost,GridAmount
0,789.79,1281.68
1,601.98,1317.69
2,616.70,1282.20


Using this multi-objective approach, we were able to **substantially reduce the total cost** compared to the initial solution, while only slightly increasing the amount of electricity purchased from the grid.

*Note:* Because the original model did not account for electricity prices, it’s possible that alternative solutions exist with the same grid usage but different overall costs.


#### Hierarchical Objectives

The hierarchical multi-objective approach prioritizes objectives in a predefined order. After optimizing the first objective, constraints are added to ensure that its value does not degrade beyond specified limits while optimizing subsequent objectives.

Two key parameters control this tolerance:

* **`abstol`**: an absolute (raw) allowable degradation in the first objective.
* **`reltol`**: a relative allowable degradation expressed as a percentage.

For example, we can first minimize total cost, then allow this cost to increase by up to 50 units or by 10% (`reltol=0.1`) while minimizing the total electricity purchased from the grid.

---

#### Modeling Battery Health via Depth of Discharge Constraints

Maintaining battery health is critical in energy storage management. One common metric is the **depth of discharge (DoD)** — the fraction of battery capacity discharged at any time.

To incorporate DoD constraints, we’ll limit how often a battery’s charge drops below a threshold (e.g., 30% of capacity).

We introduce binary variables $v_{b,t}$ that indicate whether battery $b$ is below this threshold at time $t$:

$$
v_{b,t} = \begin{cases}
1 & \text{if } s_{b,t} < \alpha \times c_b \\
0 & \text{otherwise}
\end{cases}
$$

where $s_{b,t}$ is the state of charge and $\alpha = 0.3$.

This logic can be efficiently modeled using **indicator constraints** by expressing the contrapositive:

$$
\text{If } v_{b,t} = 0, \text{ then } s_{b,t} \geq 0.3 \times c_b
$$


To respect the 2,000-variable limit imposed by the `pip`-installed Gurobi license, this constraint will only be applied to `Battery0`.
```


In [12]:
# Reminder: s_{b,t} corresponds to state[b,t] in the code

# Create binary variables to indicate if Battery0 is below 30% capacity at time t
v = m.addVars(time_periods, vtype=GRB.BINARY, name='v')

# Define a linear expression to count total depth-of-discharge violations
total_depth_count = v.sum()

# Add indicator constraints for Battery0 only:
# If v[t] == 0, then state['Battery0', t] must be >= 30% of capacity
m.addConstrs(
    ( (v[t] == 0) >> (state['Battery0', t] >= 0.3 * capacity['Battery0'])
      for t in time_periods ),
    name="discharge_depth"
)

m.update()



In [14]:
# Now we'll dive into a three-piece objective with cost as our highest priority, depth of discharge count second, and grid purchase third.
#the higher priority number is used first as an objective
m.setObjectiveN(total_cost, index=0, priority=2, reltol = 0.05, name="cost")
m.setObjectiveN(total_depth_count, index=1,  priority=1, reltol = 0.2, name= "depth")
m.setObjectiveN(total_grid, index=2, priority=0, name= "grid")
m.optimize()

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1440 rows, 1980 columns and 4498 nonzeros
Model fingerprint: 0xb4e2dd59
Model has 180 simple general constraints
  180 INDICATOR
Variable types: 1440 continuous, 540 integer (540 binary)
Coefficient statistics:
  Matrix range     [9e-01, 2e+01]
  Objective range  [6e-02, 2e+00]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e-01, 9e+01]
  GenCon rhs range [2e+01, 2e+01]
  GenCon coe range [1e+00, 1e+00]

---------------------------------------------------------------------------
Multi-objectives: starting optimization with 3 objectives... 
---------------------------------------------------------------------------

Multi-objectives: applying initial presolve...
-----------------------------------------------------------------

In [15]:
for o in range(m.NumObj):
  # set which objective we will query
  m.params.ObjNumber = o
  # query the o-th objective value
  print(' ',round(m.ObjNVal,2), end='')

  632.14  14.0  1309.58

### Handling Multiple Scenarios

In this section, we begin with our **base model**, where the objective is to minimize the total cost of electricity purchased from the grid. To facilitate easier modification and management of constraints across different scenarios, we will store key components and constraints systematically.



In [16]:
# Create a new model for multiple scenarios
mm = gp.Model('multi_scenario')

# Decision variables
flow_in = mm.addVars(batteries, time_periods, name="flow_in")
flow_out = mm.addVars(batteries, time_periods, name="flow_out")
grid = mm.addVars(time_periods, name="grid")
state = mm.addVars(
    batteries, time_periods,
    ub=[capacity[b] for b in batteries for _ in time_periods],
    name="state"
)
gen = mm.addVars(time_periods, name="gen")
zwitch = mm.addVars(batteries, time_periods, vtype=GRB.BINARY, name="zwitch")

# Expressions for total grid energy and total cost
total_grid = grid.sum()
total_cost = gp.quicksum(avg_price.price[t] * grid[t] for t in time_periods)

# Constraints
power_balance = mm.addConstrs(
    (
        gp.quicksum(flow_out[b, t] - p_loss[b] * flow_in[b, t] for b in batteries)
        + gen[t] + grid[t] == total_demand[t]
        for t in time_periods
    ),
    name="power_balance"
)

initial_state = mm.addConstrs(
    (
        state[b, 0] == initial[b] + p_loss[b] * flow_in[b, 0] - flow_out[b, 0]
        for b in batteries
    ),
    name="initial_state"
)

subsequent_states = mm.addConstrs(
    (
        state[b, t] == state[b, t-1] + p_loss[b] * flow_in[b, t] - flow_out[b, t]
        for b in batteries for t in time_periods if t >= 1
    ),
    name="subsequent_states"
)

solar_avail = mm.addConstrs(
    (
        flow_in['Battery0', t] + flow_in['Battery1', t] + gen[t] <= solar_values[t]
        for t in time_periods
    ),
    name="solar_avail"
)

to_charge = mm.addConstrs(
    (
        flow_in[b, t] <= 20 * zwitch[b, t]
        for b in batteries for t in time_periods
    ),
    name="to_charge"
)

or_not_to_charge = mm.addConstrs(
    (
        flow_out[b, t] <= 20 * (1 - zwitch[b, t])
        for b in batteries for t in time_periods
    ),
    name="or_not_to_charge"
)

# Set the objective to minimize total cost
mm.setObjective(total_cost, GRB.MINIMIZE)

# Finalize the model setup
mm.update()


#### Comparing Four Scenarios

In this section, we will evaluate four distinct scenarios. Beyond the base case described previously, the other scenarios will vary in terms of objective coefficients, variable bounds, and constraint right-hand sides (RHS). This will allow us to understand how different model parameters impact the optimization outcomes.

We begin by specifying the total number of scenarios and designate the first as our base scenario.



In [17]:
mm.NumScenarios=4
mm.Params.ScenarioNumber = 0
mm.ScenNName = 'Base model'



1. **(Mostly) Higher Costs**

For this scenario, we modified the electricity price data to amplify the variation in prices. Specifically, a new column was added to the price DataFrame where higher prices were increased further, and lower prices were decreased. The multiplier applied was randomized, resulting in more pronounced fluctuations in the price profile compared to the base case.



In [19]:
# Scenario 1: Adjusted prices with amplified variation
price2 = avg_price.price2  # Modified price column with increased volatility

# Set scenario number and name
mm.Params.ScenarioNumber = 1
mm.ScenNName = 'Increased price'

# Update objective coefficients for grid purchase variables under this scenario
for t in time_periods:
    grid[t].ScenNObj = price2[t]


2. **Lower Battery Capacity**

In this scenario, we simulate reduced battery capacities to reflect possible degradation or downsizing. The capacity of Battery0 is decreased from 60 kW to 48 kW, and Battery1 is reduced from 80 kW to 64 kW.



In [20]:
# Scenario 2: Reduced battery capacities
capacity2 = {'Battery0': 48, 'Battery1': 64}  # Updated capacities reflecting degradation

# Set scenario number and name
mm.Params.ScenarioNumber = 2
mm.ScenNName = 'Low Battery Capacity'

# Update upper bounds on battery state variables for this scenario
for b in batteries:
    for t in time_periods:
        state[b, t].ScenNUB = capacity2[b]


3. **Higher Solar Availability**

The solar forecast data includes multiple columns generated by the Prophet model, such as lower bound, point estimate, and upper bound predictions. To simulate increased solar availability, we create a new solar profile by taking a convex combination of these estimates, assigning weights that bias the forecast upwards.


In [30]:
# Scenario 3: Increased solar availability using convex combination of forecasts
solar_values2 = round(
    0.1 * solar_df.yhat_lower +
    0.6 * solar_df.yhat +
    0.3 * solar_df.yhat_upper, 3
)

# Set any negative forecast values to zero
solar_values2[solar_values2 < 0] = 0

# Set scenario number and name
mm.Params.ScenarioNumber = 3
mm.ScenNName = 'High Solar'

# Update RHS of solar availability constraints for this scenario
for t in time_periods:
    solar_avail[t].ScenNRhs = solar_values2[t]


#### Querying Scenarios

With all scenarios defined, we can export the model to an `.lp` file and run the optimization. When solving, Gurobi’s log output provides insight into how it leverages scenario information to solve efficiently—such as through warm starts, cuts, and bounds propagation across scenarios.

Try running the optimization and observe the solver log to see these performance enhancements in action.



In [31]:
# Save the model to an LP file
mm.write('ms.lp')

# Optimize the multi-scenario model
mm.optimize()

# Print objective values for each scenario
for s in range(mm.NumScenarios):
    mm.Params.ScenarioNumber = s
    print(f"\nTotal cost for scenario '{mm.ScenNName}' is ${mm.ScenNObjVal:.2f}")


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1440 rows, 1800 columns and 4498 nonzeros
Model fingerprint: 0x00b35a48
Variable types: 1440 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [9e-01, 2e+01]
  Objective range  [6e-02, 3e+00]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e-01, 9e+01]

Solving a multi-scenario model with 4 scenarios...

Found heuristic solution: objective 2722.4370000
Presolve removed 103 rows and 89 columns
Presolve time: 0.02s
Presolved: 1872 rows, 1889 columns, 5668 nonzeros
Presolved model has 4 scenario(s)
Found heuristic solution: objective 2713.0375100
Variable types: 1549 continuous, 340 integer (340 binary)
Found heuristic solution: objective 868.9568220

Root relaxation: objective 4.448080e+02, 1347 ite

### Solution Pools

Let’s explore **solution pools**, a powerful feature that enables us to find multiple feasible solutions that meet specific criteria based on the search parameters we set. This approach can be particularly useful for gaining insight into alternative strategies or for multi-objective optimization where several good solutions exist.

Dive in—the water’s fine!



In [32]:
# Configure the solution pool to collect up to 250 solutions
mp.setParam(GRB.Param.PoolSolutions, 250)

# Accept solutions within 5% of the best objective value
mp.setParam(GRB.Param.PoolGap, 0.05)

# Perform a systematic search for the k-best solutions
mp.setParam(GRB.Param.PoolSearchMode, 2)

# Run the optimization
mp.optimize()


Set parameter PoolSolutions to value 250
Set parameter PoolGap to value 0.05
Set parameter PoolSearchMode to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
PoolSolutions  250
PoolSearchMode  2
PoolGap  0.05

Optimize a model with 1440 rows, 1800 columns and 4498 nonzeros
Model fingerprint: 0x66bddde2
Variable types: 1440 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [9e-01, 2e+01]
  Objective range  [6e-02, 2e+00]
  Bounds range     [1e+00, 8e+01]
  RHS range        [1e-01, 9e+01]
Found heuristic solution: objective 2722.4370000
Presolve removed 59 rows and 59 columns
Presolve time: 0.01s
Presolved: 1381 rows, 1741 columns, 4306 nonzeros
Variable types: 1381 continuous, 360 integer (360 binary)

Root relaxation: objective 6.019761e+02, 

In [35]:
# Initialize an empty DataFrame to store flow data from multiple solutions
flow_250 = pd.DataFrame()

# Loop through all collected solutions in the solution pool
for i in range(250):
    mp.setParam(GRB.Param.SolutionNumber, i)  # Select solution i

    # Extract flow in and flow out values for each battery and time period
    tmp = pd.DataFrame(
        [
            [b, t, flow_in[b, t].Xn, flow_out[b, t].Xn, i]
            for b in batteries for t in time_periods
        ],
        columns=['battery', 'time_period', 'flow_in', 'flow_out', 'solution_index']
    )

    # Append the extracted data to the main DataFrame
    flow_250 = pd.concat([flow_250, tmp], axis=0, ignore_index=True)

flow_250




,battery,time_period,flow_in,flow_out,solution_index
0,Battery0,0,0.000,0.000000,0
1,Battery0,1,3.224,0.000000,0
2,Battery0,2,0.000,0.000000,0
3,Battery0,3,0.000,0.000000,0
4,Battery0,4,0.000,3.062800,0
...,...,...,...,...,...
89995,Battery1,175,0.000,3.530000,249
89996,Battery1,176,0.000,0.000000,249
89997,Battery1,177,0.000,0.316564,249
89998,Battery1,178,0.000,0.000000,249
